# ✈️ TRIP.com Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

> ⚠️ Trip chạy ở chế độ **BROWSER** (room API của Trip ký từng request nên không replay trực tiếp được — đã xác nhận). Chậm hơn Agoda nhưng ổn định.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/trip/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_trip.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong

In [ ]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1g_S06QeEAWnCTHYXGH0Nn4Mcb3FCT_-uIS1jUm4GGkw/edit?gid=607908359#gid=607908359"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
OFFLINE_FILE = "input/trip_hotels.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/2" = chạy phần 1 trong 2 phần (chạy lần lượt 1/2 rồi 2/2)

In [ ]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

In [ ]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "trip_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

In [ ]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "trip")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_trip.csv) nằm ở đây

kwargs = dict(
    site="trip",                     # browser-per-query (Trip không replay trực tiếp được)
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

In [ ]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "trip")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)